In [1]:
import zmq
import queue
import threading
import time
import uuid
import cloudpickle

context = zmq.Context()

# Clients
socket_client = context.socket(zmq.REP)
socket_client.bind("tcp://*:5555")

# Worker0000
socket_worker = context.socket(zmq.REQ)
socket_worker.connect("tcp://192.168.0.102:5556")

file = queue.Queue()

jobs = {}

def generate_job_name():
    return f"job_{uuid.uuid4().hex[:8]}"

jobs_lock = threading.Lock()
nombre_demandes=0
nombre_demandes_realisees=0

In [2]:
def cleanup_jobs():

    while True:

        time.sleep(540)
        print("suppression des items finis depuis plus de 15min dans 1min")
        time.sleep(60)
        now = int(time.time())
        jobs_to_delete = []
        with jobs_lock:
            for nom, job in jobs.items():

                if job["status"] == "cancelled":
                    jobs_to_delete.append(nom)
                elif (job["status"] in ["done", "failed"]  and job["heure de fin"] is not None    and (now - job["heure de fin"]) > 15*60):
                    jobs_to_delete.append(nom)
        for nom in jobs_to_delete:
            with jobs_lock:
                del jobs[nom]
        print(f"Nettoyage : {len(jobs_to_delete)} jobs supprimés")

In [3]:
def dispatch_worker():
    global nombre_demandes_realisees
    while True:
        job = file.get()
        nom = job["nom"]

        # Job annulé avant son exécution
        
        with jobs_lock:
            if jobs[nom]["status"] == "cancelled":
                del jobs[nom]
                nombre_demandes_realisees-=1 #pour avoir le bon compte car déjà annulé avant
                continue
            jobs[nom]["status"] = "running"
            jobs[nom]["heure début"] = int(time.time())

        print(f"Envoi au worker : {nom}")
        
        socket_worker.send(
            cloudpickle.dumps(job))
        result = cloudpickle.loads(socket_worker.recv())
        nombre_demandes_realisees+=1
        print(f"Résultat reçu : {nom}")
        with jobs_lock:
            if nom not in jobs: continue
            jobs[nom]["status"] = result["status"]
            jobs[nom]["result"] = result["result"]
            jobs[nom]["heure de fin"]=int(time.time())
            if jobs[nom]["status"]=="failed":
                jobs[nom]["error_type"]=result["error_type"]
        print(f"{nom} est stocké")

In [4]:
socket_worker.send(cloudpickle.dumps({"type": "get_config"}))
soc_config = cloudpickle.loads(socket_worker.recv())

print("Configuration QICK chargée")

Configuration QICK chargée


In [5]:
worker_thread = threading.Thread(
    target=dispatch_worker,
    daemon=True
)

cleanup_thread = threading.Thread(
    target=cleanup_jobs,
    daemon=True
)

worker_thread.start()
cleanup_thread.start()

Envoi au worker : my_job
Résultat reçu : my_job
my_job est stocké
Envoi au worker : job_643bdc2b
Résultat reçu : job_643bdc2b
Envoi au worker : job_3274bcdc
Résultat reçu : job_3274bcdc
job_3274bcdc est stocké
Envoi au worker : job_2d2338ee
Résultat reçu : job_2d2338ee
job_2d2338ee est stocké
Envoi au worker : job_60e08115
Résultat reçu : job_60e08115
job_60e08115 est stocké
Envoi au worker : job_655c856d
Résultat reçu : job_655c856d
job_655c856d est stocké
Envoi au worker : job_60e25642
Résultat reçu : job_60e25642
job_60e25642 est stocké
Envoi au worker : job_cfd18c13
Résultat reçu : job_cfd18c13
job_cfd18c13 est stocké
Envoi au worker : job_31959939
Résultat reçu : job_31959939
job_31959939 est stocké
Envoi au worker : job_e1414c8c
Résultat reçu : job_e1414c8c
job_e1414c8c est stocké
Envoi au worker : job_f6d89338
Résultat reçu : job_f6d89338
job_f6d89338 est stocké
Envoi au worker : job_3ac7d633
Résultat reçu : job_3ac7d633
job_3ac7d633 est stocké
suppression des items finis depuis

In [ ]:
while True:

    request = cloudpickle.loads(socket_client.recv())
    if "client_id" in request :
        print("demande reçue de", request["client_id"])
    if "type" not in request :
        socket_client.send(cloudpickle.dumps({
                "status": "failed",
                "result": "failed to search",
                "error_type": "format de requête invalide"}))
        continue
    typ = request["type"]
    heure_arrive=int(time.time())
    
    if typ == "STOP":
        socket_client.send(cloudpickle.dumps("Serveur : arrêt"))
        break

    
    elif typ == "get_config":
        nombre_demandes+=1
        nombre_demandes_realisees+=1
        socket_client.send(cloudpickle.dumps(soc_config))
        continue

        
    elif typ == "Requete":
        nombre_demandes+=1
        proposed_name = request["nom"]
        
            # Aucun nom fourni
        if proposed_name is None:
            nom = generate_job_name()
            message = (
            f"Aucun nom fourni. "
            f"Le job a été enregistré sous '{nom}'.")

        # Nom fourni et disponible
        elif proposed_name not in jobs:
            nom = proposed_name
            message = (f"Job '{nom}' ajouté à la file.")
        # Nom déjà utilisé
        elif jobs[proposed_name]["status"] == "cancelled": 
            nom = proposed_name
            message = (f"Job '{nom}' ajouté à la file.")
            with jobs_lock:
                jobs[nom] = {
                    "status": "queued",
                    "position_queue":nombre_demandes,
                    "result": None,
                    "client_id": request["client_id"],
                    "heure d'arrive":heure_arrive,
                    "heure début":None,
                    "heure de fin":None}
            print(f"Ajout FIFO : {nom}")#reste dans la file au même emplacement malgré tout, étonnant mais fonctionne
            socket_client.send(cloudpickle.dumps({"message": message,"nom": nom}))
            continue
        else:
            nom = generate_job_name()
            message = (f"Le nom '{proposed_name}' était déjà utilisé. "
            f"Le job a été renommé en '{nom}'.")
        request["nom"] = nom
        with jobs_lock:
            jobs[nom] = {
                "status": "queued",
                "position_queue":nombre_demandes,
                "result": None,
                "client_id": request["client_id"],
                "heure d'arrive":heure_arrive,
                "heure début":None,
                "heure de fin":None}
        file.put(request)
        print(f"Ajout FIFO : {nom}")
        socket_client.send(cloudpickle.dumps({"message": message,"nom": nom}))
        
    
    elif typ == "cancel":
        nombre_demandes+=1
        nom = request["nom"]
        client_id = request["client_id"]
        with jobs_lock:
            if nom in jobs:
                nombre_demandes_realisees+=1
                if jobs[nom]["client_id"] != client_id and not client_id.startswith("admin"):
                    socket_client.send(cloudpickle.dumps(f"Vous n'êtes pas autorisé à annuler le job {nom}"))
                    continue
                else:
                    socket_client.send(cloudpickle.dumps(f"Job {nom} annulé"))
                if jobs[nom]["status"]!="queued":
                    del jobs[nom]
                else:
                    jobs[nom]["status"] = "cancelled"
                print(f"Nettoyage : {nom} supprimé")
            else:
                socket_client.send(cloudpickle.dumps(f"La requête {nom} n'a pas pu être annulée"))


    elif typ == "recuperer":
        nom = request["nom"]
        if nom not in jobs:
            socket_client.send(cloudpickle.dumps({
                "status": "failed",
                "result": "failed to search",
                "error_type": "Job inconnu"}))
        else:
            nombre_demandes+=1
            with jobs_lock:
                element = jobs[nom].copy()
            if element["status"]=="done":
                socket_client.send(cloudpickle.dumps({
                "status": element["status"],
                "result": element["result"],
                "temps total": element["heure de fin"] -  element["heure d'arrive"],
                "durée taitement" : (jobs[nom]["heure de fin"]- jobs[nom]["heure début"])}))
            elif element["status"]=="failed":
                socket_client.send(cloudpickle.dumps({
                "status": element["status"],
                "result": element["result"],
                "error_type": element["error_type"]}))
            elif element["status"]=="queued":
                socket_client.send(cloudpickle.dumps({
                "status": element["status"],
                "result": f"demande en position {(element['position_queue']-nombre_demandes_realisees-1)}"}))
            else: socket_client.send(cloudpickle.dumps({
                "status": element["status"],
                "result": element["result"]}))
            nombre_demandes_realisees+=1
    else : socket_client.send(cloudpickle.dumps({
                "status": "failed",
                "result": "failed to search",
                "error_type": "format de requête invalide"}))
    

C:\Users\NS2-manip\AppData\Local\miniforge3\envs\qick\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
Ajout FIFO : my_job
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
Ajout FIFO : job_643bdc2b
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
Nettoyage : job_643bdc2b supprimé
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
Ajout FIFO : job_3274bcdc
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
Ajout FIFO : job_2d2338ee
demande reçue de 7dc9405a-108a-4938-a6de-81130891abb9
Ajout FIFO : job_60e08115
demande reçue de aa00ca2f-3ced-4d61-8a49-ec1c984c7670
Ajout FIFO : job_655c856d
demande reçue de aa00ca2f-3ced-4d61-8a49-ec1c984c7670
demande reçue de aa00ca2f-3ced-4d61-8a49-ec1c984c7670
demand